# Phase 8 : Système de Recommandation (E-commerce)

Ce notebook implémente un moteur de recommandation hybride pour suggérer des produits aux clients en fonction de leur historique d'achat.

## Objectifs :
- Construire une matrice d'interaction Utilisateur-Produit.
- Calculer les similarités entre clients (Collaborative Filtering).
- Recommander des produits 'Next Best Offer' pour un client donné.

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import joblib
import os

import warnings
warnings.filterwarnings('ignore')

In [2]:
# Chargement des données d'interaction
df = pd.read_csv('../data/processed/dataset_ml_features.csv')

# Création de la matrice User-Item (par fréquence d'achat)
user_item_matrix = df.groupby(['FK_Client', 'FK_Produit']).size().unstack(fill_value=0)

print(f"Matrice User-Item : {user_item_matrix.shape[0]} clients, {user_item_matrix.shape[1]} produits")
user_item_matrix.head()

Matrice User-Item : 267 clients, 32 produits


FK_Produit,0,6,15,16,24,26,34,36,38,39,...,76,80,83,87,93,95,99,103,110,112
FK_Client,,,,,,,,,,,,,,,,,,,,,
0,35,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 1. Filtrage Collaboratif (User-Based)

Nous calculons la similarité cosinus entre les vecteurs d'achats des clients.

In [3]:
# Calcul de la similarité cosinus entre utilisateurs
user_similarity = cosine_similarity(user_item_matrix)
user_similarity_df = pd.DataFrame(user_similarity, index=user_item_matrix.index, columns=user_item_matrix.index)
user_similarity_df.head()

FK_Client,0,1,2,3,4,5,6,7,8,9,...,308,309,310,311,312,314,315,317,318,319
FK_Client,,,,,,,,,,,,,,,,,,,,,
0,1.000000,0.0,0.999592,0.999592,0.0,0.0,0.999592,0.0,0.02856,0.999592,...,0.0,0.999592,0.0,0.0,0.999592,0.999592,0.0,0.0,0.999592,0.0
1,0.000000,1.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.00000,0.000000,...,0.0,0.000000,0.0,1.0,0.000000,0.000000,1.0,0.0,0.000000,0.0
2,0.999592,0.0,1.000000,1.000000,0.0,0.0,1.000000,0.0,0.00000,1.000000,...,0.0,1.000000,0.0,0.0,1.000000,1.000000,0.0,0.0,1.000000,0.0
3,0.999592,0.0,1.000000,1.000000,0.0,0.0,1.000000,0.0,0.00000,1.000000,...,0.0,1.000000,0.0,0.0,1.000000,1.000000,0.0,0.0,1.000000,0.0
4,0.000000,0.0,0.000000,0.000000,1.0,0.0,0.000000,0.0,0.00000,0.000000,...,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0


In [4]:
def get_recommendations(client_id, matrix, similarity_df, n_recommendations=3):
    if client_id not in matrix.index:
        return []
    
    # Trouver les utilisateurs les plus similaires
    similar_users = similarity_df[client_id].sort_values(ascending=False)[1:6].index
    
    # Produits achetés par ces utilisateurs mais pas par le client actuel
    client_purchases = matrix.loc[client_id]
    unbought_products = client_purchases[client_purchases == 0].index
    
    # Score des produits basés sur les achats des voisins
    reco_scores = matrix.loc[similar_users, unbought_products].mean().sort_values(ascending=False)
    
    return reco_scores.head(n_recommendations).index.tolist()

# Test
sample_client = user_item_matrix.index[0]
print(f"Recommandations pour le client {sample_client} : {get_recommendations(sample_client, user_item_matrix, user_similarity_df)}")

Recommandations pour le client 0 : [6, 15, 110]


In [5]:
# Sauvegarde pour l'application Web
os.makedirs('../models', exist_ok=True)
joblib.dump(user_item_matrix, '../models/user_item_matrix.joblib')
joblib.dump(user_similarity_df, '../models/user_similarity_df.joblib')

print("Modèles de recommandation sauvegardés.")

Modèles de recommandation sauvegardés.
